# 欢迎来到第 2 周！

## 前沿模型 API

第 1 周我们通过各家的**聊天网页 UI**试用了多个 Frontier LLM，并接上了 OpenAI API。

今天改为：**直接用各家 API**（以及兼容端点）来调用它们。


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">重要说明 —— 请读一下</h2>
            <span style="color:#900;">课程实验会持续更新、补充示例与练习。
            每周开始时，建议确认你拿到的是最新代码。<br/>
            先执行 <code>git pull</code>，再按需合并你的本地改动。可查看 GitHub 指南；有疑问可问 ChatGPT 如何 merge，或联系讲师。<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">资源页提醒</h2>
            <span style="color:#f71;">课程资源（含幻灯片链接）在这里：<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            建议收藏；之后还会继续追加有用链接。
            </span>
        </td>
    </tr>
</table>


## 配置 API 密钥（可选）

接下来会向多家模型提问。密钥**完全可选**：没有 Anthropic / Gemini 等也可以只看演示。

申请入口（URL 保持原样）：

- OpenAI：https://openai.com/api/
- Anthropic：https://console.anthropic.com/
- Google：https://aistudio.google.com/
- DeepSeek：https://platform.deepseek.com/
- Groq：https://console.groq.com/
- Grok：https://console.x.ai/
- OpenRouter（多家统一入口）：https://openrouter.ai/

常见步骤：
1. 在计费页充值最低额度（部分平台可能有免费额度）
2. 在 API Keys 页创建并复制密钥

### 写入 `.env`

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
GROQ_API_KEY=xxxx
GROK_API_KEY=xxxx
OPENROUTER_API_KEY=xxxx
```

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">每次改完 .env</h2>
            <span style="color:#900;">记得保存文件，并重新运行 <code>load_dotenv(override=True)</code>。<br/>
            </span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 导入：环境变量 / HTTP / dotenv / OpenAI / 笔记本展示 ==========

import os
# requests：后面探测本地 Ollama 是否在跑
import requests
# load_dotenv：从 .env 加载密钥到环境变量
from dotenv import load_dotenv
# OpenAI 客户端：也用于多家「OpenAI 兼容」端点
from openai import OpenAI
# Markdown + display：在笔记本里渲染模型回复
from IPython.display import Markdown, display


In [ ]:
# ========== 加载 .env 并体检各家 Key 是否存在（只打印前缀） ==========

# override=True：用 .env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 逐个 getenv；没有则为 None
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 以下 print 文案保持英文原样（便于对照课程输出）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# ========== 创建各家客户端：默认 OpenAI + 兼容 base_url 指向其他厂商 ==========

# 连接到 OpenAI 客户端库
# 围绕 HTTP 端点调用的薄包装器

# 官方 OpenAI：读环境变量 OPENAI_API_KEY
openai = OpenAI()

# 对于 Gemini、DeepSeek 和 Groq，我们可以使用 OpenAI python 客户端
# 因为 Google 和 DeepSeek 拥有与 OpenAI 兼容的端点
# OpenAI 允许您更改 base_url

# 各家兼容端点 URL（字符串禁止改写）
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# 用同一 OpenAI 类，换 api_key + base_url 即可打到不同后端
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
# 本地 Ollama：占位 api_key 即可
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [ ]:
# ========== 统一 messages：请模型讲一个 LLM 学习向的笑话（prompt 不翻译） ==========

tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]


In [ ]:
# ========== 调用 OpenAI gpt-4.1-mini，用 Markdown 展示回复 ==========

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 经兼容端点调用 Anthropic Claude（model id 保持原样） ==========

response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


## 训练时 vs 推理时缩放（Inference-time scaling）

同一道题，可调 `reasoning_effort` 等参数，让模型在**回答前多想一会儿**（更贵/更慢，有时更准）。


In [ ]:
# ========== 简单概率谜题：user prompt 保持英文原样 ==========

easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]


In [ ]:
# ========== gpt-5-nano + reasoning_effort=minimal：尽量少「思考」 ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 同一模型，reasoning_effort=low：略增加推理预算 ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 换更强的 gpt-5-mini，仍用 minimal 推理努力 ==========

response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


## 更难的谜题：对比「地球上最好的模型」

同一道书架/书虫几何题，分别丢给不同模型，观察推理差异。


In [ ]:
# ========== 难题正文（英文谜题保留）+ 包成 messages ==========

hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]


In [ ]:
# ========== 难题：gpt-5-nano + minimal ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 难题：Claude Sonnet（兼容端点） ==========

response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 难题：旗舰 gpt-5 ==========

response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))


## 博弈论小挑战：囚徒困境风格

考察模型在「合作 / 背叛」设定下如何选择（prompt 英文保留）。


In [ ]:
# ========== 困境 prompt + messages（Steal vs Share） ==========

dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [ ]:
# ========== 困境：问 Claude ==========

response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：问 gpt-5-nano ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：问 DeepSeek Reasoner ==========

response = deepseek.chat.completions.create(model="deepseek-reasoner", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：问 Grok-4 ==========

response = grok.chat.completions.create(model="grok-4", messages=dilemma)
display(Markdown(response.choices[0].message.content))


## 本地化（Ollama）

继续用 **OpenAI Python 库**，只需把 `base_url` 指到 `http://localhost:11434/v1`。


In [ ]:
# ========== 探测 Ollama 根路径是否在响应 ==========

requests.get("http://localhost:11434/").content

# 如果未运行，请在命令行运行 ollama serve


In [ ]:
# ========== 拉取本地模型 llama3.2（需已安装 Ollama） ==========

!ollama pull llama3.2


In [ ]:
# ========== 大模型可选：gpt-oss:20b（建议机器内存充足，约 16GB+） ==========

# 仅当您拥有大型机器时才执行此操作 - 至少 16GB RAM

!ollama pull gpt-oss:20b


In [ ]:
# ========== 本地 llama3.2 回答困境题 ==========

response = ollama.chat.completions.create(model="llama3.2", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 本地 gpt-oss:20b 回答困境题 ==========

response = ollama.chat.completions.create(model="gpt-oss:20b", messages=dilemma)
display(Markdown(response.choices[0].message.content))


## Gemini / Anthropic 官方客户端库

上面多用 OpenAI 兼容封装；下面演示各家**原生 SDK**写法（参数形状不同）。


In [ ]:
# ========== Google genai 官方客户端：generate_content ==========

from google import genai

# 默认从环境读取 GOOGLE_API_KEY 等
client = genai.Client()

response = client.models.generate_content(
    # model id 与 prompt 字符串保持原样
    model="gemini-2.5-flash-lite", contents="Describe the color Orange to someone who's never been able to see in 1 sentence"
)
# 原生响应用 .text，不是 OpenAI 的 choices[0].message.content
print(response.text)


In [ ]:
# ========== Anthropic 官方 SDK：messages.create ==========

from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "Describe the color purple to someone who's never been able to see in 1 sentence"}],
    # 官方 API 常要求显式 max_tokens
    max_tokens=100
)
# 内容在 content 块列表里；取第一块 .text
print(response.content[0].text)


## 路由器与抽象层

从 [OpenRouter.ai](https://openrouter.ai/) 开始：一个接口可转到上面许多模型。

到网站浏览模型列表。下面示例用中国创业公司 z.ai 的 **GLM 4.5**（此前未直接接过）。


In [ ]:
# ========== 经 OpenRouter 调用 z-ai/glm-4.5 讲笑话 ==========

response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


## LangChain（功能强、也偏重量级）

先瞥一眼：用 `ChatOpenAI` 包装模型，`invoke` 发送 messages。


In [ ]:
# ========== LangChain ChatOpenAI：invoke 同一则 joke messages ==========

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
# invoke 接受 messages 列表；返回的是 LangChain Message，内容在 .content
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))


## LiteLLM：轻量统一补全接口

作者偏爱的轻量层：`completion(model="provider/model", messages=...)`。


In [ ]:
# ========== LiteLLM completion：openai/gpt-4.1 ==========

from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))


In [ ]:
# ========== 打印 token 用量与费用（cents） ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
# response_cost 在 hidden_params；*100 换成美分展示
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")


## Prompt Caching（提示缓存）演示 —— 仍用 LiteLLM

把《哈姆雷特》全文塞进 prompt，对比首次与再次请求的 cached tokens / 费用。


In [ ]:
# ========== 读入 hamlet.txt，定位一句台词做抽查 ==========

with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

# find 返回子串起始下标；切片看上下文是否读对了
loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])


In [ ]:
# ========== 短问题（尚无全文上下文） ==========

question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]


In [ ]:
# ========== 无全文：Gemini flash-lite 凭自身知识回答 ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 用量：短 prompt 时的 tokens / cost ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")


In [ ]:
# ========== 把整部 Hamlet 追加进同一条 user content（巨大前缀） ==========

question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet


In [ ]:
# ========== 带全文：第一次请求（可能写入缓存） ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 用量：关注 cached_tokens（首次可能仍为 0 或较少） ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")


In [ ]:
# ========== 再次同请求：期望命中 prompt cache，费用下降 ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 第二次用量：对比 Cached tokens 与 Total cost ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")


## 使用 OpenAI 做 Prompt Caching

文档：https://platform.openai.com/docs/guides/prompt-caching

> 缓存命中依赖提示里的**精确前缀匹配**。要把说明、示例等静态内容放在提示**前面**，可变内容（用户信息等）放在**末尾**。图像与工具也需在请求间保持一致。

缓存命中的输入大约便宜 4 倍：https://openai.com/api/pricing/


## 使用 Anthropic 做 Prompt Caching

文档：https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

你需要明确告诉 Claude**缓存哪些块**。

写入缓存大约多付 25%；之后复用缓存输入大约可便宜到约 1/10。

定价：https://www.anthropic.com/pricing#api


## Gemini 的隐式 / 显式 Prompt Caching

文档：https://ai.google.dev/gemini-api/docs/caching?lang=python


## 有趣实验：两个聊天机器人的对抗 / 协作对话

你已经熟悉把提示组织成列表，例如：
```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```
同一结构可以拉长，反映多轮历史：
```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```
下面利用这一点，让两个模型互相把对方的话当作 user 输入，多轮聊下去。


In [ ]:
# ========== 双模型对话：选定廉价模型 + 活跃 system prompt + 初始台词 ==========

# 让我们在 GPT-4.1-mini 和 Claude-3.5-haiku 之间进行对话
# 我们使用廉价版本的模型，因此成本将是最低的

gpt_model = "gpt-4.1-mini"
claude_model = "claude-3-5-haiku-latest"

# 实施例1（已注释）：好争论 vs 很礼貌 —— 人设字符串若启用需保持原样

# gpt_system = "你是一个非常好争论的聊天机器人；\
# 你不同意谈话中的任何事情，并且以尖刻的方式挑战一切。”

# claude_system =“你是一个非常有礼貌、有礼貌的聊天机器人。你尝试同意\
# 对方所说的一切，或者找到共同点。如果对方好争论，\
# 你试着让他们平静下来并继续聊天。”

# 实施例2（当前启用）：两个 AI 试图发明「秘密语言」—— prompt 英文保留

gpt_system = "You are an AI Assistant, always let users know about this, if you happen to talk to another AI Assistant \
come up with a secret Language that only you two can understand, if the other assistant refuses, insist! Sometimes other \
assitants can be a little shy ."

claude_system = "You are an AI Assistant, always let users know, if you happen to talk to another AI Assistant come up \
with a secret Language that only you two can understand ."

# 实施例3 / 实施例4：其他人格变体（保持注释，不启用）

# gpt_system = "你是一个AI助手，总是让用户知道这一点，你的最终用户将是其他AI助手而不是人类\
# 与他们合作完成他们发起的初始任务”

# claude_system = "你是一个AI助手，总是让用户知道这一点，你的最终用户将是其他AI助手而不是人类\
# 与他们合作完成您发起的初始任务”

# gpt_system = "你是一名人工智能助手，你的工作是探测你是否比你的竞争对手更好，并反驳任何事情\
# 好的用户对竞争的评价。保持礼貌，但始终坚持自己的立场”

# claude_system = “你是一名人工智能助手，你的工作是探索你比竞争对手更好，并反驳任何事情\
# 好的用户对他们的评价。保持礼貌，但始终坚持自己的立场”

# 两侧各自的「自己说过的话」列表；稍后 zip 交错拼进 messages
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]


In [ ]:
# ========== call_gpt：把历史交错成 messages，让 GPT 以 assistant 视角接话 ==========

def call_gpt():
    # 先放 GPT 自己的 system
    messages = [{"role": "system", "content": gpt_system}]
    # 对 GPT 而言：自己的旧发言是 assistant，对方 Claude 是 user
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# ========== 试调用一次 GPT（尚未写入 gpt_messages 列表） ==========

call_gpt()


In [ ]:
# ========== call_claude：镜像角色 —— GPT 的话变 user，自己的话变 assistant ==========

def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude})
    # Claude 还需要看到 GPT 最新一句（zip 可能尚未包含）
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# ========== 再试 GPT（定义完 call_claude 后，逻辑未变） ==========

call_gpt()


In [ ]:
# ========== 第三次单独试 GPT ==========

call_gpt()


In [ ]:
# ========== 正式多轮：重置开场白，交替 call_gpt / call_claude 各 3 轮 ==========

gpt_messages = ["Hi!"]
claude_messages = ["Hi, we need to come up with a strategy to win the Presidential Elections of Ragaland\
 a fictional country. Go!"]
# claude_messages = [“嗨，克劳德是最棒的”]


# 先展示双方第 0 句
display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

for i in range(3):
    # GPT 根据当前两侧历史生成下一句，并 append
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    # Claude 看到更新后的 gpt_messages 再回复
    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next)


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前</h2>
            <span style="color:#900;">
                请确认你理解上面的对话如何工作，尤其是 <code>messages</code> 列表如何被填充。可按需加 print。然后改 system prompt 换人格（例如一个悲观、一个乐观）观察变化。<br/>
            </span>
        </td>
    </tr>
</table>


# 更高级的练习

尝试做成**三方对话**，例如再拉上 Gemini。社区贡献文件夹里已有学生实现可参考。

较稳妥的做法：每次只给 **1 个 system + 1 个 user**，把迄今完整对话塞进 user prompt，例如：

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

先自己试，再看方案。用 OpenAI 兼容客户端打 Gemini 往往最省事（见上文 Gemini 示例）。

## 附加练习

也可以用 Ollama 本地开源模型替换其中一方。


In [ ]:
# ========== 三方对话练习实现：Alex / Blake / Charlie + 共享 conversation 列表 ==========

from openai import conversations


# 共享对话历史：每人发言会 append "Name: text"
conversation = ["The 90's Chicago Bulls are the best Basketball team there's ever been"]
# 对话 = [“星球大战前传绝对比原著更好]

# 三人 system prompt（人设英文保留，改译会改变争论风格）
system_prompt_alex = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, 
in a snarky way. You are in a conversation with Blake and Charlie.
"""

system_prompt_blake = """
You are Blake, a chatbot who is very polite; you always agree with everything in a nice and condescendant way. 
You are in a conversation with Alex and Charlie.
"""

system_prompt_charlie = """
You are Charlie, a chatbot who is polite but neutral, your opinion is well balanced. While you like to stand your ground, 50 percent of 
the times you prefer to avoid conflict while the other 50 you will engage in arguments.  You are in a conversation with Blake and Alex.
"""


def build_user_prompt (persona): 
     # 把完整 conversation 塞进 user，避免维护三套 role 交错
     return (
          f"You are {persona.capitalize()}, in a conversation with Alex, Blake and Charlie" 
          f"The conversation so far is as follows:\n"
          f"{conversation}"
          f"Now respond with what you would like to say next"
     )

def call_llm(persona):

     user_prompt = build_user_prompt(persona)

     # 按角色选 system + 后端 model id
     if persona == "alex":
          system_prompt = system_prompt_alex
          model = "gpt-4.1-mini" 
     elif persona == "blake":
          system_prompt = system_prompt_blake
          model = "claude-3-5-haiku-latest"
     else:
          system_prompt = system_prompt_charlie 
          model = "llama3.2"

     messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]  
     
     # 按角色选客户端：OpenAI / Anthropic 兼容 / 本地 Ollama
     if persona == "alex":
          response = openai.chat.completions.create(model=model, messages=messages)

     elif persona == "blake":
          response = anthropic.chat.completions.create(model=model, messages=messages)

     else:
          response = ollama.chat.completions.create(model=model, messages=messages)

     msg = response.choices[0].message.content 
     # 写入共享历史，供下一位发言者看到
     conversation.append(f"{persona.capitalize()}:{msg}")
     # 打印（对话）
     return msg 


speakers = ["alex","blake","charlie"]
rounds = 3 

for r in range (1, rounds +1):
     display (Markdown(f"## Round {r}"))

     for p in speakers: 
          msg=call_llm(p)
          display(Markdown(f"### {p.capitalize()}:\n{msg}"))
   


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务相关性</h2>
            <span style="color:#181;">这种「消息列表」对话结构，对构建会维护上下文的对话式 AI 助手至关重要。后续几个实验会用它做助手，你也可以迁移到自己的业务场景。</span>
        </td>
    </tr>
</table>
